# PaddleOCR-VL 1.6 (0.9B VLM) — Kaggle T4 Benchmark

Compact vision-language document-parsing model (0.9B). SOTA 96.33% on OmniDocBench v1.6. Runs on the native **Paddle** engine — NOT vLLM — because the T4 (compute 7.5) is below vLLM's 8.0 requirement.

**Kaggle setup (do this first):**
1. Settings → **Accelerator** → `GPU T4 x2` (the notebook uses a single T4 = `gpu:0`).
2. Settings → **Internet** → **On** (needed for `pip install` and model download).
3. Set `BASE_URL` in the next cell to where your test images are hosted, then **Run All**.

This notebook measures **GPU memory (peak)**, **disk usage** (model cache + outputs),
**per-image latency**, prints **model info**, saves **JSON + annotated images (+ Markdown)**,
displays every output figure inline, and bundles everything into a **downloadable ZIP**.

In [ ]:
# ============================ CONFIG ============================
# Images come from an attached Kaggle Dataset. The loader searches INPUT_DIR
# RECURSIVELY, so it works whether the images sit directly under this path or
# inside a nested "image" subfolder.
INPUT_DIR = "/kaggle/input/datasets/amasikifthakerhamim/images"

# Optional: instead of a Kaggle Dataset, download images from a URL host.
# Leave BASE_URL = "" to use the Kaggle Dataset above.
BASE_URL    = ""
IMAGE_NAMES = [f"image_{i:03d}.jpg" for i in range(1, 12)]    # only used when BASE_URL is set

OUTPUT_DIR  = "/kaggle/working/output_paddleocr_vl_1.6"
ZIP_PATH    = "/kaggle/working/paddleocr_vl_1.6_results"   # .zip appended automatically

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================ INSTALL ============================
# PaddleOCR (doc-parser extras) + GPU PaddlePaddle 3.2.1 (CUDA 12.6 wheel).
# Runs on the T4 (compute 7.5 >= paddle's required 7.0).
#
# WHY WE REMOVE torch BELOW (important):
#   paddlepaddle-gpu (cu126) downgrades nvidia-*-cu12 to 12.6, which BREAKS Kaggle's
#   preinstalled torch (built for cu128 -> "undefined symbol: ncclCommShrink"). And
#   `import paddleocr` indirectly imports torch via modelscope's logger. These notebooks
#   use PaddlePaddle only, so we simply uninstall the torch stack: modelscope then detects
#   torch is absent (find_spec) and skips it, and paddleocr imports cleanly.
#   (--timeout/--retries make the slow Paddle mirror resilient to the occasional timeout.)
!pip install -q "paddleocr[doc-parser]"
!pip install -q "paddlepaddle-gpu==3.2.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ --timeout 180 --retries 10
!pip install -q nvidia-ml-py
# Remove the now-broken, unused torch stack LAST so nothing re-adds it before we import paddle.
!pip -q uninstall -y torch torchvision torchaudio
print("\nInstall finished. torch removed on purpose (unused here); the earlier pip 'conflict' lines are resolved by this.")

In [ ]:
# ============================ VERIFY GPU ============================
# Confirms PaddlePaddle actually sees the T4 before we spend time loading models.
import paddle
print("paddlepaddle :", paddle.__version__)
print("CUDA compiled:", paddle.device.is_compiled_with_cuda())
print("GPU count    :", paddle.device.cuda.device_count())
assert paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() >= 1, (
    "PaddlePaddle does not see a GPU. Set Accelerator = 'GPU T4 x2', enable Internet, "
    "then Run > Restart & Run All."
)
# quick end-to-end GPU sanity op
_x = paddle.to_tensor([1.0, 2.0, 3.0], place=paddle.CUDAPlace(0))
print("GPU tensor sum:", float((_x * 2).sum()), "-> PaddlePaddle GPU is working.")

# Import paddleocr now so any dependency issue surfaces here (not mid-run).
# (torch was uninstalled above, so the modelscope->torch import chain is skipped.)
import importlib
assert importlib.util.find_spec("torch") is None, (
    "torch is still installed and will break the paddleocr import via modelscope. "
    "Re-run the INSTALL cell (it uninstalls torch), then Restart & Run All."
)
import paddleocr
print("paddleocr    :", paddleocr.__version__, "-> import OK.")

In [ ]:
# ============================ GET IMAGES ============================
import os, urllib.request, glob

EXTS = ("jpg", "jpeg", "png", "bmp", "webp", "jfif", "tif", "tiff")

def find_local_images(root):
    got = []
    for e in EXTS:                       # recursive: handles a nested image/ folder
        got += glob.glob(os.path.join(root, "**", f"*.{e}"), recursive=True)
        got += glob.glob(os.path.join(root, "**", f"*.{e.upper()}"), recursive=True)
    return sorted(set(got))

def fetch_images():
    if BASE_URL:                         # download-from-URL mode
        dl = "/kaggle/working/images"; os.makedirs(dl, exist_ok=True)
        base = BASE_URL.rstrip("/")
        for n in IMAGE_NAMES:
            dst = os.path.join(dl, n)
            if not os.path.exists(dst):
                try: urllib.request.urlretrieve(f"{base}/{n}", dst)
                except Exception as ex: print("skip", n, "->", ex)
        return find_local_images(dl)
    return find_local_images(INPUT_DIR)  # Kaggle Dataset mode

images = fetch_images()
assert images, (f"No images found under {INPUT_DIR!r}. Check the dataset is attached "
                "(Add Input) and the path is correct, or set BASE_URL to a URL host.")
print(f"{len(images)} images found under {INPUT_DIR if not BASE_URL else BASE_URL}:")
for p in images:
    print("  ", os.path.relpath(p, INPUT_DIR) if not BASE_URL else os.path.basename(p))

In [ ]:
# ============================ BENCH UTILS ============================
import os, time, threading, subprocess, gc

try:
    import pynvml
    pynvml.nvmlInit()
    _H = pynvml.nvmlDeviceGetHandleByIndex(0)   # single T4 = index 0
    _NVML = True
except Exception as e:
    _NVML = False
    print("pynvml unavailable, GPU memory will read NaN:", e)

def gpu_used_mb():
    if not _NVML:
        return float("nan")
    return pynvml.nvmlDeviceGetMemoryInfo(_H).used / 1024**2

class GPUSampler:
    """Context manager: polls GPU memory and records the peak used (MB)."""
    def __init__(self, interval=0.05):
        self.interval, self.peak, self._run, self._t = interval, 0.0, False, None
    def _loop(self):
        while self._run:
            try: self.peak = max(self.peak, gpu_used_mb())
            except Exception: pass
            time.sleep(self.interval)
    def __enter__(self):
        self.peak = gpu_used_mb() if _NVML else 0.0
        self._run = True
        self._t = threading.Thread(target=self._loop, daemon=True); self._t.start()
        return self
    def __exit__(self, *a):
        self._run = False
        if self._t: self._t.join()

def dir_size_mb(path):
    path = os.path.expanduser(path); total = 0
    for root, _, files in os.walk(path):
        for f in files:
            try: total += os.path.getsize(os.path.join(root, f))
            except OSError: pass
    return total / 1024**2

MODEL_CACHE_DIRS = ["~/.paddlex", "~/.paddleocr", "~/.cache/huggingface", "~/.cache/paddle"]
def model_cache_mb():
    return sum(dir_size_mb(d) for d in MODEL_CACHE_DIRS)

def show_env():
    q = "name,memory.total,driver_version,compute_cap"
    out = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    print("GPU:", out)

show_env()

In [ ]:
# ============================ MODEL INFO + LOAD ============================
import paddle, paddleocr, json
print("paddlepaddle:", paddle.__version__, "| CUDA compiled:", paddle.device.is_compiled_with_cuda())
print("paddleocr   :", paddleocr.__version__)

MODEL_INFO = {
    'model': 'PaddleOCR-VL-1.6-0.9B',
    'type': 'Vision-Language document parser (layout + VLM)',
    'params': '~0.9B (ERNIE-4.5-0.3B decoder + NaViT-style encoder)',
    'benchmark': 'OmniDocBench v1.6 = 96.33 (overall)',
    'engine': 'paddle (native) — vLLM NOT used (T4 CC 7.5 < 8.0)',
    'capabilities': 'text, tables, formulas, charts, seals, reading order',
}
print(json.dumps(MODEL_INFO, indent=2))

cache_before = model_cache_mb()
t0 = time.time()
with GPUSampler() as g_load:
    from paddleocr import PaddleOCRVL
    
    # Native Paddle engine on a single T4 (gpu:0). vLLM/FlashAttention are NOT
    # used because the T4 is compute-capability 7.5 (vLLM needs >= 8.0).
    predictor = PaddleOCRVL(pipeline_version='v1.6', device='gpu:0')
load_time = time.time() - t0
model_disk_mb = model_cache_mb() - cache_before
gpu_after_load = gpu_used_mb()
print(f"\nLoad time      : {load_time:.1f} s")
print(f"Model download : {model_disk_mb:.1f} MB")
print(f"GPU after load : {gpu_after_load:.0f} MB (peak during load {g_load.peak:.0f} MB)")

In [ ]:
# ============================ RUN INFERENCE ============================
SAVE_FNS = ['save_to_json', 'save_to_img', 'save_to_markdown']
per_image = []

with GPUSampler() as g_run:
    for p in images:
        name = os.path.basename(p)
        t0 = time.time()
        results = list(predictor.predict(str(p)))
        dt = time.time() - t0
        for res in results:
            for fn in SAVE_FNS:
                try: getattr(res, fn)(save_path=OUTPUT_DIR)
                except Exception as e: print(f"  {fn} failed on {name}: {e}")
        per_image.append({"image": name, "seconds": round(dt, 3), "regions": len(results)})
        print(f"  {name:<28} {dt:6.2f} s")

peak_gpu_run = g_run.peak
avg_latency = sum(r['seconds'] for r in per_image) / len(per_image)
print(f"\nAvg latency : {avg_latency:.2f} s/image")
print(f"Peak GPU    : {peak_gpu_run:.0f} MB")

In [ ]:
# ============================ SHOW OUTPUT FIGURES ============================
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg

viz = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.jpg"), recursive=True) +
             glob.glob(os.path.join(OUTPUT_DIR, "**", "*.png"), recursive=True))
print(f"{len(viz)} visualization figures produced.")

for f in viz[:24]:                       # cap inline display at 24 figures
    img = mpimg.imread(f)
    h, w = img.shape[:2]
    plt.figure(figsize=(min(14, w/80), min(18, h/80)))
    plt.imshow(img); plt.axis("off"); plt.title(os.path.relpath(f, OUTPUT_DIR), fontsize=9)
    plt.show()
if len(viz) > 24:
    print(f"... {len(viz)-24} more figures saved in the ZIP.")

In [ ]:
# ============================ SHOW JSON OUTPUT ============================
import glob, json
jsons = sorted(glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True))
print(f"{len(jsons)} JSON files produced. Preview of the first:\n")
if jsons:
    with open(jsons[0], "r", encoding="utf-8") as f:
        data = json.load(f)
    txt = json.dumps(data, indent=2, ensure_ascii=False)
    print(txt[:4000] + ("\n... (truncated, full JSON is in the ZIP)" if len(txt) > 4000 else ""))

In [ ]:
# ============================ RESOURCE SUMMARY ============================
import json, pandas as pd

output_disk_mb = dir_size_mb(OUTPUT_DIR)
summary = {
    "model": MODEL_INFO["model"],
    "num_images": len(images),
    "gpu_peak_MB": round(peak_gpu_run, 1),
    "gpu_after_load_MB": round(gpu_after_load, 1),
    "avg_latency_s": round(avg_latency, 3),
    "total_infer_s": round(sum(r['seconds'] for r in per_image), 2),
    "model_disk_MB": round(model_disk_mb, 1),
    "output_disk_MB": round(output_disk_mb, 1),
    "load_time_s": round(load_time, 1),
    "engine": MODEL_INFO.get("engine", "paddle"),
}
with open(os.path.join(OUTPUT_DIR, "benchmark_summary.json"), "w") as f:
    json.dump({"summary": summary, "model_info": MODEL_INFO, "per_image": per_image}, f, indent=2)

print("=== RESOURCE SUMMARY ===")
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
print("\n=== PER-IMAGE LATENCY ===")
display(pd.DataFrame(per_image))

In [ ]:
# ============================ ZIP EVERYTHING ============================
import shutil, os
from IPython.display import FileLink

zip_file = shutil.make_archive(ZIP_PATH, "zip", OUTPUT_DIR)
size_mb = os.path.getsize(zip_file) / 1024**2
print(f"Wrote {zip_file}  ({size_mb:.1f} MB)")
print("Contents: annotated images, JSON, markdown (if any), benchmark_summary.json")
FileLink(zip_file)   # click to download from the Kaggle output panel